In [92]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [93]:
#df = pd.read_csv("nexlyra_tbm_fraud_1M_ravi_ml.csv",low_memory=False)

In [94]:
file_path = "nexlyra_tbm_fraud_1M_ravi_ml.csv"

###  Initialize Python and load the 1M-row dataset. Notice how much RAM it consumes.

In [95]:
import os
# Check file size on disk
file_size_mb = os.path.getsize(file_path) / (1024 ** 2)
print(f"CSV file size: {file_size_mb:.2f} MB")

CSV file size: 241.21 MB


In [96]:
# Load dataset
df = pd.read_csv(file_path)

In [97]:
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Shape: (1000000, 7)
Rows: 1,000,000
Columns: 7


In [98]:
# Check RAM consumption
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"RAM used by DataFrame: {memory_mb:.2f} MB")
print(f"RAM used by DataFrame: {memory_mb / 1024:.2f} GB")

RAM used by DataFrame: 253.44 MB
RAM used by DataFrame: 0.25 GB


### The 'Nested_TBM_JSON' column contains deeply nested dictionaries.

In [99]:
df.head()

,Log_ID,TBM_Timestamp,Tunnel_Section,Contractor_Name,Nested_TBM_JSON,Target_Advance_Rate,Is_Spoofed
0,TBM-LOG-0106693,2023-03-16T08:39:00Z,Metro_Line_B,Global_Tunnels,"{""tbm_meta"": {""sensor_hex"": ""54424d2d53454e532...",5.628410,0
1,TBM-LOG-0624846,2024-03-10T04:06:00Z,Undersea_Cable_Core,Phantom_Boring_LLC,"{""tbm_meta"": {""sensor_hex"": ""54424d2d53454e532...",3.568811,0
2,TBM-LOG-0483853,2023-12-03T02:27:00Z,Metro_Line_A,Apex_Drilling,"{""tbm_meta"": {""sensor_hex"": ""54424d2d53454e532...",2.819102,0
3,TBM-LOG-0531196,2024-01-05T04:22:00Z,Metro_Line_B,Phantom_Boring_LLC,"{""tbm_meta"": {""sensor_hex"": ""54424d2d53454e532...",3.977614,0
4,TBM-LOG-0248328,06/22/2023 07:17:00,Undersea_Cable_Core,Phantom_Boring_LLC,"{""tbm_meta"": {""sensor_hex"": ""54424d2d53454e532...",4.266647,0


In [100]:
df['Nested_TBM_JSON']

0         {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
1         {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
2         {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
3         {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
4         {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
                                ...                        
999995    {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
999996    {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
999997    {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
999998    {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
999999    {"tbm_meta": {"sensor_hex": "54424d2d53454e532...
Name: Nested_TBM_JSON, Length: 1000000, dtype: str

In [101]:
print(df.loc[0, 'Nested_TBM_JSON'])

{"tbm_meta": {"sensor_hex": "54424d2d53454e532d36303834", "torque_knm": 542.75, "hydraulics": {"pressure_bar": 180.48, "thermal_c": 45.91}}, "status": "OK"}


### How do I efficiently parse a 1-Million row nested JSON column in pandas without running out of memory using chunking?'

In [102]:
import json

In [103]:
chunk_size = 50000
results = []
for chunk in pd.read_csv(file_path,chunksize=chunk_size,usecols=["Log_ID","TBM_Timestamp","Tunnel_Section","Contractor_Name",
                                                                 "Nested_TBM_JSON","Target_Advance_Rate","Is_Spoofed"]):
    
    # Parse JSON only for the current chunk
    parsed = chunk["Nested_TBM_JSON"].apply(lambda x: json.loads(x) if pd.notna(x) else {})
    
    # Extract only the fields you actually need
    chunk["sensor_hex"] = parsed.apply(lambda x: x.get("tbm_meta", {}).get("sensor_hex"))

    chunk["torque_knm"] = parsed.apply(lambda x: x.get("tbm_meta", {}).get("torque_knm"))

    chunk["pressure_bar"] = parsed.apply(lambda x: x.get("tbm_meta", {}).get("hydraulics", {}).get("pressure_bar"))

    chunk["thermal_c"] = parsed.apply(lambda x: x.get("tbm_meta", {}).get("hydraulics", {}).get("thermal_c"))

    chunk["status"] = parsed.apply(lambda x: x.get("status"))

    # Don't keep the huge nested JSON after extraction
    chunk.drop(columns=["Nested_TBM_JSON"], inplace=True)

    results.append(chunk)

###  Apply the AI's optimized solution to extract 'sensor_hex', 'torque_knm', 'pressure_bar', and 'thermal_c'.

In [104]:
final_df = pd.concat(results, ignore_index=True)
print(final_df.shape)

(1000000, 11)


In [105]:
final_df

,Log_ID,TBM_Timestamp,Tunnel_Section,Contractor_Name,Target_Advance_Rate,Is_Spoofed,sensor_hex,torque_knm,pressure_bar,thermal_c,status
0,TBM-LOG-0106693,2023-03-16T08:39:00Z,Metro_Line_B,Global_Tunnels,5.628410,0,54424d2d53454e532d36303834,542.75,180.48,45.91,OK
1,TBM-LOG-0624846,2024-03-10T04:06:00Z,Undersea_Cable_Core,Phantom_Boring_LLC,3.568811,0,54424d2d53454e532d39373335,896.92,230.74,42.84,OK
2,TBM-LOG-0483853,2023-12-03T02:27:00Z,Metro_Line_A,Apex_Drilling,2.819102,0,54424d2d53454e532d31383239,1164.14,151.75,51.13,OK
3,TBM-LOG-0531196,2024-01-05T04:22:00Z,Metro_Line_B,Phantom_Boring_LLC,3.977614,0,54424d2d53454e532d31383035,882.03,181.52,45.21,OK
4,TBM-LOG-0248328,06/22/2023 07:17:00,Undersea_Cable_Core,Phantom_Boring_LLC,4.266647,0,54424d2d53454e532d34323737,615.47,NaN,NaN,WARN
...,...,...,...,...,...,...,...,...,...,...,...
999995,TBM-LOG-0424816,10/23/2023 04:48:00,HighSpeed_Rail_Link,Global_Tunnels,4.168224,0,54424d2d53454e532d32313138,515.21,234.15,41.81,OK
999996,TBM-LOG-0279949,2023-07-14T06:18:00Z,Metro_Line_B,Titan_Excavators,3.769479,0,54424d2d53454e532d35323135,835.56,224.07,40.92,OK
999997,TBM-LOG-0857667,2024-08-18T18:22:00Z,Undersea_Cable_Core,Phantom_Boring_LLC,1.530402,0,54424d2d53454e532d32303037,1150.76,280.20,41.72,OK
999998,TBM-LOG-0942053,2024-10-16T10:13:00Z,Metro_Line_B,Phantom_Boring_LLC,3.199701,0,54424d2d53454e532d36313935,833.74,225.76,44.66,OK


### Handle the missing 'hydraulics' blocks dynamically, filling missing pressure/thermal with np.nan.

In [106]:
final_df.isnull().sum()

Log_ID                      0
TBM_Timestamp               0
Tunnel_Section              0
Contractor_Name             0
Target_Advance_Rate     25000
Is_Spoofed                  0
sensor_hex              20077
torque_knm              20077
pressure_bar           167274
thermal_c              167274
status                  20077
dtype: int64

In [107]:
final_df["pressure_bar"] = final_df["pressure_bar"].replace("", np.nan)
final_df["thermal_c"] = final_df["thermal_c"].replace("", np.nan)

In [108]:
final_df.isnull().sum()

Log_ID                      0
TBM_Timestamp               0
Tunnel_Section              0
Contractor_Name             0
Target_Advance_Rate     25000
Is_Spoofed                  0
sensor_hex              20077
torque_knm              20077
pressure_bar           167274
thermal_c              167274
status                  20077
dtype: int64

In [109]:
print(final_df["pressure_bar"].unique())
print(final_df["thermal_c"].unique())

[180.48 230.74 151.75 ... 943.67 939.49 861.86]
[45.91 42.84 51.13 ... 54.78 57.35 52.01]


In [110]:
print(final_df[["pressure_bar", "thermal_c"]].isna().sum())

pressure_bar    167274
thermal_c       167274
dtype: int64


In [111]:
print(final_df.loc[final_df["pressure_bar"].isna(), ["pressure_bar", "thermal_c"]].head())

    pressure_bar  thermal_c
4            NaN        NaN
5            NaN        NaN
13           NaN        NaN
14           NaN        NaN
16           NaN        NaN


- We no need to fill thease columns with NaN value because it is already filled with NaN values.

### The 'TBM_Timestamp' column has mixed formats (UTC vs Local). Use pd.to_datetime safely.

In [112]:
final_df["TBM_Timestamp"] = pd.to_datetime(final_df["TBM_Timestamp"],errors="coerce")

In [113]:
final_df["TBM_Timestamp"].dtypes

datetime64[us, UTC]

### Sort the entire DataFrame strictly by Tunnel_Section and TBM_Timestamp to restore the time-series sequence.

In [114]:
final_df.head()

,Log_ID,TBM_Timestamp,Tunnel_Section,Contractor_Name,Target_Advance_Rate,Is_Spoofed,sensor_hex,torque_knm,pressure_bar,thermal_c,status
0,TBM-LOG-0106693,2023-03-16 08:39:00+00:00,Metro_Line_B,Global_Tunnels,5.628410,0,54424d2d53454e532d36303834,542.75,180.48,45.91,OK
1,TBM-LOG-0624846,2024-03-10 04:06:00+00:00,Undersea_Cable_Core,Phantom_Boring_LLC,3.568811,0,54424d2d53454e532d39373335,896.92,230.74,42.84,OK
2,TBM-LOG-0483853,2023-12-03 02:27:00+00:00,Metro_Line_A,Apex_Drilling,2.819102,0,54424d2d53454e532d31383239,1164.14,151.75,51.13,OK
3,TBM-LOG-0531196,2024-01-05 04:22:00+00:00,Metro_Line_B,Phantom_Boring_LLC,3.977614,0,54424d2d53454e532d31383035,882.03,181.52,45.21,OK
4,TBM-LOG-0248328,NaT,Undersea_Cable_Core,Phantom_Boring_LLC,4.266647,0,54424d2d53454e532d34323737,615.47,NaN,NaN,WARN


In [115]:
final_df = final_df.sort_values(by=["Tunnel_Section", "TBM_Timestamp"],ascending=[True, True]).reset_index(drop=True)

In [116]:
print(final_df[["Tunnel_Section", "TBM_Timestamp"]].head(10))

        Tunnel_Section             TBM_Timestamp
0  HighSpeed_Rail_Link 2023-01-01 00:00:00+00:00
1  HighSpeed_Rail_Link 2023-01-01 00:05:00+00:00
2  HighSpeed_Rail_Link 2023-01-01 00:12:00+00:00
3  HighSpeed_Rail_Link 2023-01-01 00:24:00+00:00
4  HighSpeed_Rail_Link 2023-01-01 00:27:00+00:00
5  HighSpeed_Rail_Link 2023-01-01 00:29:00+00:00
6  HighSpeed_Rail_Link 2023-01-01 00:33:00+00:00
7  HighSpeed_Rail_Link 2023-01-01 00:44:00+00:00
8  HighSpeed_Rail_Link 2023-01-01 00:48:00+00:00
9  HighSpeed_Rail_Link 2023-01-01 00:59:00+00:00


### 'Write a vectorized pandas function to convert a column of hex-encoded strings (sensor_hex) back to ASCII.'

In [117]:
final_df["sensor_hex"] = final_df["sensor_hex"].apply(lambda x: bytes.fromhex(x).decode("ascii") if pd.notna(x) else np.nan)

In [118]:
final_df["sensor_hex"]

0         TBM-SENS-5410
1         TBM-SENS-8004
2         TBM-SENS-8906
3         TBM-SENS-2693
4         TBM-SENS-1710
              ...      
999995    TBM-SENS-8391
999996    TBM-SENS-8298
999997    TBM-SENS-7431
999998    TBM-SENS-1889
999999    TBM-SENS-4191
Name: sensor_hex, Length: 1000000, dtype: str

### Decode the Sensor Hashes. Verify the expected output format.

In [119]:
print(final_df["sensor_hex"].str.match(r"^TBM-SENS-\d{4}$", na=False).all())

False


In [120]:
pattern = r"^TBM-SENS-\d{4}$"
invalid = final_df.loc[~final_df["sensor_hex"].str.match(pattern, na=False),"sensor_hex"]
print(invalid.head(20))

67     NaN
75     NaN
283    NaN
374    NaN
384    NaN
399    NaN
406    NaN
439    NaN
482    NaN
550    NaN
585    NaN
586    NaN
694    NaN
702    NaN
822    NaN
836    NaN
855    NaN
869    NaN
920    NaN
951    NaN
Name: sensor_hex, dtype: str


In [121]:
print(final_df["sensor_hex"].str.match(r"^TBM-SENS-\d{4}$", na=False).all())

False


- After varification We found that after decoding column sensor_hex contains NaN values.
- Hence Regex formate match showing False.

### Target_Advance_Rate contains NaN values. Do NOT use simple dropna() as it breaks the time-series continuity.

In [122]:
print(final_df["Target_Advance_Rate"].isna().sum())

25000


In [123]:
print(final_df["Target_Advance_Rate"].dtypes)

float64


In [124]:
final_df["Target_Advance_Rate"] = final_df["Target_Advance_Rate"].interpolate(method="linear")

In [125]:
print(final_df["Target_Advance_Rate"].isna().sum())

0


- Since it has null values and belongs to timeseries features.
- We would better to interpolate by linear after varifying it is numeric values.

### Perform tensor memory profiling on DataFrame chunk #1 to ensure dtype downcasting (float64 to float32) was successful

In [126]:
print(final_df.dtypes)

Log_ID                                 str
TBM_Timestamp          datetime64[us, UTC]
Tunnel_Section                         str
Contractor_Name                        str
Target_Advance_Rate                float64
Is_Spoofed                           int64
sensor_hex                             str
torque_knm                         float64
pressure_bar                       float64
thermal_c                          float64
status                                 str
dtype: object


In [127]:
print(final_df.select_dtypes(include="float64").columns)

Index(['Target_Advance_Rate', 'torque_knm', 'pressure_bar', 'thermal_c'], dtype='str')


In [129]:
memory_before = final_df.memory_usage(deep=True).sum() / 1024**2

In [130]:
float_cols = final_df.select_dtypes(include="float64").columns
final_df[float_cols] = final_df[float_cols].astype("float32")

In [131]:
memory_after = final_df.memory_usage(deep=True).sum() / 1024**2

In [132]:
print(f"Memory before: {memory_before:.2f} MB")
print(f"Memory after:  {memory_after:.2f} MB")
print(f"Memory saved:  {memory_before - memory_after:.2f} MB")

Memory before: 142.44 MB
Memory after:  127.18 MB
Memory saved:  15.26 MB


In [133]:
final_df.head()

,Log_ID,TBM_Timestamp,Tunnel_Section,Contractor_Name,Target_Advance_Rate,Is_Spoofed,sensor_hex,torque_knm,pressure_bar,thermal_c,status
0,TBM-LOG-0000002,2023-01-01 00:00:00+00:00,HighSpeed_Rail_Link,Apex_Drilling,5.425898,0,TBM-SENS-5410,616.229980,157.660004,40.500000,OK
1,TBM-LOG-0000007,2023-01-01 00:05:00+00:00,HighSpeed_Rail_Link,Phantom_Boring_LLC,2.993575,0,TBM-SENS-8004,726.419983,288.799988,62.570000,OK
2,TBM-LOG-0000014,2023-01-01 00:12:00+00:00,HighSpeed_Rail_Link,Titan_Excavators,2.742072,0,TBM-SENS-8906,1071.800049,163.740005,43.000000,OK
3,TBM-LOG-0000023,2023-01-01 00:24:00+00:00,HighSpeed_Rail_Link,Phantom_Boring_LLC,4.853213,0,TBM-SENS-2693,733.659973,158.250000,41.849998,OK
4,TBM-LOG-0000027,2023-01-01 00:27:00+00:00,HighSpeed_Rail_Link,Global_Tunnels,3.534157,0,TBM-SENS-1710,849.030029,262.470001,57.200001,OK


In [134]:
final_df.to_csv("final_df.csv", index=False)

In [ ]:
- Same the final out to csv file
- Convert all float64 into float32 in single attempt.
- Need to preprocess more to the next step